In [ ]:
%pip install paho-mqtt pyserial
%pip  install ipywidgets
%pip -q install pyserial
%pip install jupyter_http_over_ws

In [1]:
#Port verifiaction to check that the microbit is detected

import serial.tools.list_ports as list_ports

for p in list_ports.comports():
    print(f"Device: {p.device}")
    print(f"  Description: {p.description}")
    print(f"  VID: {getattr(p, 'vid', None)}")
    print(f"  PID: {getattr(p, 'pid', None)}")
    print()

Device: COM5
  Description: USB Serial Device (COM5)
  VID: 3368
  PID: 516



In [2]:
import paho.mqtt.client as mqtt
import serial
import time
import sys
import threading
import json

# === CONFIGURE THESE ===
BROKER = "broker.hivemq.com"
PORT = 1883
DATA_TOPIC = "microbit/data"
CMD_TOPIC = "microbit/commands"

SERIAL_PORT = "COM5"       # Adjust port depending on the outputs from las cell
BAUD_RATE = 115200

# H0 - The Physical I/O Layer

# Standard identifiers for the BBC micro:bit
VID_MICROBIT = 3368
PID_MICROBIT = 516
BAUD = 115200
_ser = None

def find_port():
    """Detects the micro:bit COM port based on VID/PID or Description."""
    for p in list_ports.comports():
        if getattr(p, 'vid', None) == VID_MICROBIT and getattr(p, 'pid', None) == PID_MICROBIT:
            return p.device
    for p in list_ports.comports():
        d = (p.description or '').lower()
        if any(x in d for x in ['micro:bit', 'mbed', 'daplink', 'bbc']):
            return p.device
    return None

def open_serial():
    """Opens the serial port if it isn't already open."""
    global _ser
    if _ser and _ser.is_open:
        return _ser
    port = find_port()
    if not port:
        raise RuntimeError('micro:bit not found. Check USB connection and ensure you are on a Local Runtime.')
    _ser = serial.Serial(port, BAUD, timeout=0.2)
    time.sleep(0.2) # Allow for hardware handshake
    return _ser

In [3]:
find_port()
open_serial()

Serial<id=0x2de1251b970, open=True>(port='COM5', baudrate=115200, bytesize=8, parity='N', stopbits=1, timeout=0.2, xonxoff=False, rtscts=False, dsrdtr=False)

In [4]:
def on_connect(client, userdata, flags, rc, properties=None):
    print(f"✅ HiveMQ Connected - Code: {rc} ({mqtt.connack_string(rc)})")
    if rc == 0:
        client.subscribe(CMD_TOPIC)
        print(f"   📡 Subscribed to: {CMD_TOPIC}")

def on_message(client, userdata, msg):
    command = msg.payload.decode().strip()
    print(f"📥 Command from Colab: {command}")
    
    if command == "FEAR":
        print("🚨 FEAR command received! Triggering hardware...")
        _ser.write(b'FEAR\n')
        _ser.flush()
        
    else:
        _ser.write((command + "\n").encode())
        _ser.flush()


client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="local_microbit")
client.on_connect = on_connect
client.on_message = on_message

client.connect(BROKER, PORT, 60)
client.loop_start()   # starts MQTT network loop in background

<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

✅ HiveMQ Connected - Code: Success (Success)
   📡 Subscribed to: microbit/commands


Code below is still being tested.

In [ ]:

def read_serial_and_publish():
    while True:
        if _ser.in_waiting > 0:
            line = _ser.readline().decode("utf-8").strip()
            if line:
                print(f"Data from micro:bit: {line}")
                # Publish raw data (string or JSON) to Colab
                client.publish(DATA_TOPIC, line)
        time.sleep(0.05)   # small delay to avoid CPU spin

# Start reading in a background thread
thread = threading.Thread(target=read_serial_and_publish, daemon=True)
thread.start()
print("Serial reader thread started – notebook is now live")

In [ ]:
# === TEST: Send fake sensor data from Local to Colab ===
test_messages = [
    "temp:35",
    "light:40",
    "temp:25",
    "button:pressed",
    "temp:42"
]

for msg in test_messages:
    print(f"📤 Publishing: {msg}")
    client.publish(DATA_TOPIC, msg)
    time.sleep(2)   # give Colab time to process

📤 Publishing: temp:35
📥 Command from Colab: FAN_ON
📤 Publishing: light:40
📥 Command from Colab: LED_ON
📤 Publishing: temp:25
📥 Command from Colab: LED_OFF
📤 Publishing: button:pressed
📥 Command from Colab: LED_OFF
📤 Publishing: temp:42
📥 Command from Colab: FAN_ON


📥 Command from Colab: FEAR
🚨 FEAR command received! Triggering hardware...
📥 Command from Colab: FEAR
🚨 FEAR command received! Triggering hardware...
